In [0]:
%pip install "psycopg[binary]>=3.1.0" -q

In [0]:
# ── Sync Lakebase → Delta (Unity Catalog) ──────────────────────
# Sincroniza las tablas de la app cfe-notas-obra desde Lakebase
# hacia Delta tables en dipidev.default para analytics.
#
# Tablas sincronizadas:
#   - notas_obra      → dipidev.default.notas_obra_sync
#   - bitacora_app    → dipidev.default.bitacora_app_sync
#   - obras_catalogo  → dipidev.default.obras_catalogo_sync
#
# Programar como Job: cada 15 min, 1 hora, o según necesidad.
# ────────────────────────────────────────────────────────────────

import psycopg
from databricks.sdk import WorkspaceClient
from pyspark.sql.types import *
from datetime import datetime

w = WorkspaceClient()

# ── Lakebase connection ────────────────────────────────────────
LAKEBASE_ENDPOINT = "projects/lakebaseworkshop/branches/production/endpoints/primary"
PGHOST = "ep-lively-mountain-eb5kgl2c.database.southcentralus.azuredatabricks.net"
PGDATABASE = "databricks_postgres"

def get_pg_conn():
    cred = w.postgres.generate_database_credential(endpoint=LAKEBASE_ENDPOINT)
    me = w.current_user.me()
    return psycopg.connect(
        host=PGHOST, dbname=PGDATABASE, user=me.user_name,
        port=5432, password=cred.token, sslmode="require",
    )

# ── Target catalog/schema ──────────────────────────────────────
CATALOG = "dipidev"
SCHEMA = "default"

# ── Sync functions ─────────────────────────────────────────────
def sync_table(query, schema, table_name):
    conn = get_pg_conn()
    with conn.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
    conn.close()
    target = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.createDataFrame(rows, schema=schema)
    df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(target)
    print(f"  {target}: {len(rows)} rows synced")
    return len(rows)

print(f"Sync started at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

# ── 1. notas_obra ──────────────────────────────────────────────
notas_schema = StructType([
    StructField("id", LongType()), StructField("id_obra", IntegerType()),
    StructField("nombre_obra", StringType()), StructField("tipo_nota", StringType()),
    StructField("descripcion", StringType()), StructField("prioridad", StringType()),
    StructField("evidencia_url", StringType()), StructField("usuario", StringType()),
    StructField("fecha_registro", TimestampType()), StructField("estado", StringType()),
    StructField("fecha_estado", TimestampType()),
])
sync_table(
    """SELECT id, id_obra, nombre_obra, tipo_nota, descripcion, prioridad,
              evidencia_url, usuario, fecha_registro,
              COALESCE(estado, 'Abierta') as estado, fecha_estado
       FROM notas_obra ORDER BY id""",
    notas_schema, "notas_obra_sync"
)

# ── 2. bitacora_app ────────────────────────────────────────────
bitacora_schema = StructType([
    StructField("id", LongType()), StructField("accion", StringType()),
    StructField("detalle", StringType()), StructField("usuario", StringType()),
    StructField("ip_cliente", StringType()), StructField("fecha", TimestampType()),
])
sync_table(
    "SELECT id, accion, detalle, usuario, ip_cliente, fecha FROM bitacora_app ORDER BY id",
    bitacora_schema, "bitacora_app_sync"
)

# ── 3. obras_catalogo ──────────────────────────────────────────
obras_schema = StructType([
    StructField("id_obra", IntegerType()), StructField("nombre_obra", StringType()),
])
sync_table(
    "SELECT id_obra, nombre_obra FROM obras_catalogo ORDER BY id_obra",
    obras_schema, "obras_catalogo_sync"
)

print("=" * 50)
print(f"Sync completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")